In [0]:
print('Silver Products')

### Data Reading

In [0]:
df = spark.read.format('parquet')\
    .load('abfss://bronze@datalakeete1.dfs.core.windows.net/products')

In [0]:
df.display()

In [0]:
df=df.drop('_rescued_data')

### ***Functions***

In [0]:
df.createOrReplaceTempView('v_products')

In [0]:
%sql
select * from v_products limit 1;

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cat_1234.bronze.discount_func(p_price DOUBLE)
RETURNS DOUBLE
LANGUAGE SQL
RETURN p_price * 0.90

In [0]:
%sql
select product_id,price,
databricks_cat_1234.bronze.discount_func(price) as discounted_price from v_products; 

In [0]:
from pyspark.sql.functions import expr
df = df.withColumn('discounted_price',expr('databricks_cat_1234.bronze.discount_func(price)'))

In [0]:
df.display()

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cat_1234.bronze.upper_func(p_brand STRING)
RETURNS STRING
LANGUAGE PYTHON
AS
$$
    return p_brand.upper()
$$

In [0]:
%sql
SELECT product_id,brand,databricks_cat_1234.bronze.upper_func(brand) as brand_upper
FROM v_products;

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .option('path','abfss://silver@datalakeete1.dfs.core.windows.net/products')\
    .save()

In [0]:
df=spark.read.format('delta')\
    .load('abfss://silver@datalakeete1.dfs.core.windows.net/products')

In [0]:
df.display()

In [0]:
%sql
CREATE TABLE databricks_cat_1234.silver.products_silver
USING DELTA
LOCATION 'abfss://silver@datalakeete1.dfs.core.windows.net/products'

In [0]:
%sql
select * from databricks_cat_1234.silver.products_silver limit 1;